Automatically generated by Colab.

Original file is located at
    https://colab.research.google.com/drive/1J1GPGTl1GfiPNrym4rJXD15sGlI84z8k

# ⚡ Transmission Line Fault Detection System
### Random Forest Classifier · Feature Analysis · XAI Dashboard
---
**Run cells in order (Shift+Enter or Runtime → Run all)**

| Cell | What it does |
|------|-------------|
| 1 | Install packages |
| 2 | Core imports & styling |
| 3 | Upload datasets |
| 4 | Data preview & class distribution |
| 5 | Feature configuration (interactive) |
| 6 | Feature analysis (Heatmap / VIF / RFE) |
| 7 | Train Random Forest + evaluate |
| 8 | ROC curves + Learning curve |
| 9 | Select prediction source |
| 10 | Execute predictions & summary plots |
| 11 | SHAP Explainability |
| 12 | Interactive SCADA dashboard |
| 13 | Export results |

## Cell 1 — Install dependencies

In [ ]:
import subprocess, sys
for pkg in ['shap', 'statsmodels']:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pkg])
print('✅ All packages ready')


## Cell 2 — Core imports & global state

In [ ]:
import os, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output
from google.colab import files
from io import BytesIO
from itertools import cycle
from datetime import datetime, timedelta

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, learning_curve
from sklearn.metrics import (accuracy_score, classification_report,
                             confusion_matrix, roc_curve, auc)
from sklearn.preprocessing import MinMaxScaler, LabelEncoder, label_binarize
from sklearn.feature_selection import RFE
from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.tools.tools import add_constant
import shap

warnings.filterwarnings('ignore')
plt.rcParams.update({
    'figure.facecolor': '#1a1a2e', 'axes.facecolor': '#16213e',
    'axes.edgecolor': '#444466', 'axes.labelcolor': '#c8c8e8',
    'xtick.color': '#888899', 'ytick.color': '#888899',
    'text.color': '#e0e0f0', 'grid.color': '#2a2a4a',
    'grid.linewidth': 0.6, 'axes.titlecolor': '#e0e0f0',
    'axes.titlesize': 13, 'axes.labelsize': 11,
    'legend.facecolor': '#1a1a2e', 'legend.edgecolor': '#444466',
    'legend.labelcolor': '#e0e0f0',
})

STATE = {
    'train_df': None, 'test_df': None, 'synthetic_df': None, 'pmu_df': None,
    'model': None, 'scaler': None, 'features': [], 'target_col': None,
    'label_enc': None, 'predictions': None,
    'X_val': None, 'y_val': None, 'y_pred': None,
}
LABEL_MAP    = {0:'Normal A', 1:'Normal B', 2:'LG Fault',
                3:'LL Fault', 4:'LLG Fault', 5:'Healthy'}
FAULT_CLASSES = [2, 3, 4]
PALETTE       = ['#00d4ff','#f72585','#4cc9f0','#7209b7','#3a0ca3','#4361ee']

def hdr(text, color='#00d4ff'):
    display(HTML(
        f"<div style='background:linear-gradient(90deg,#0f0f2d,#1a1a3e);"
        f"border-left:4px solid {color};border-radius:6px;"
        f"padding:10px 18px;margin:16px 0 8px;"
        f"font-family:monospace;font-size:14px;color:{color};'>"
        f"⚡ {text}</div>"))

def kpi_row(metrics: dict):
    colors = ['#00d4ff','#f72585','#4cc9f0','#7209b7','#3fb950']
    cards  = ''.join(
        f"<div style='flex:1;background:#0f0f2d;border:1px solid {colors[i%5]}33;"
        f"border-bottom:3px solid {colors[i%5]};border-radius:8px;"
        f"padding:16px;margin:0 6px;text-align:center;'>"
        f"<div style='font-size:1.6rem;font-family:monospace;color:{colors[i%5]};'>{v}</div>"
        f"<div style='font-size:.72rem;color:#888;text-transform:uppercase;"
        f"letter-spacing:.08em;margin-top:4px;'>{k}</div></div>"
        for i,(k,v) in enumerate(metrics.items()))
    display(HTML(f"<div style='display:flex;margin:10px 0;'>{cards}</div>"))

print('✅ Imports complete')


## Cell 3 — Upload datasets

In [ ]:
# Upload any/all of: train_dataset.csv · test_dataset.csv
#                    synthetic_fault_data_*.csv · dataset.csv
# ══════════════════════════════════════════════════════════════════
display(HTML(
    "<div style='background:#0f0f2d;border:1px dashed #00d4ff55;border-radius:10px;"
    "padding:18px 22px;font-family:monospace;color:#00d4ff;margin-bottom:10px;'>"
    "<b>📂 Dataset Upload</b><br>"
    "<span style='color:#888;font-size:12px;'>Select any combination of CSV files.<br>"
    "Accepted: train_dataset.csv · test_dataset.csv · "
    "synthetic_fault_data_*.csv · dataset.csv</span></div>"))

uploaded = files.upload()

ROUTING = {
    'train':     ['train', 'train_dataset'],
    'test':      ['test',  'test_dataset'],
    'synthetic': ['synthetic', 'fault_data'],
    'pmu':       ['dataset', 'pmu', 'raw'],
}
def route_file(fname):
    fl = fname.lower().replace('.csv','')
    for key, kws in ROUTING.items():
        if any(kw in fl for kw in kws):
            return key
    return None

for fname, content in uploaded.items():
    df   = pd.read_csv(BytesIO(content))
    slot = route_file(fname)
    if slot == 'train':     STATE['train_df']     = df
    elif slot == 'test':    STATE['test_df']      = df
    elif slot == 'synthetic': STATE['synthetic_df'] = df
    elif slot == 'pmu':     STATE['pmu_df']       = df
    else:
        if 'Fault_Type' in df.columns and STATE['train_df'] is None:
            STATE['train_df'] = df; slot = 'train (auto)'
        elif 'fault_type' in df.columns and STATE['pmu_df'] is None:
            STATE['pmu_df']   = df; slot = 'pmu (auto)'
        else:
            STATE['test_df']  = df; slot = 'test (auto)'
    print(f'  ✅  {fname}  →  {slot}  ({len(df):,} rows × {df.shape[1]} cols)')

loaded = {k: (STATE[k+'_df'] is not None) for k in ['train','test','synthetic','pmu']}
display(HTML(
    "<div style='margin-top:12px;font-family:monospace;font-size:13px;'>"
    + ''.join(
        f"<span style='background:{'#1a3a1a' if v else '#2a1a1a'};"
        f"color:{'#3fb950' if v else '#888'};border-radius:4px;"
        f"padding:3px 10px;margin:3px 4px;display:inline-block;'>"
        f"{'✓' if v else '○'} {k}</span>"
        for k,v in loaded.items())
    + "</div>"))


## Cell 4 — Data preview & class distribution

In [ ]:
named = {'Training Data': STATE['train_df'], 'Test Data': STATE['test_df'],
         'Synthetic (historical)': STATE['synthetic_df'], 'PMU Waveform': STATE['pmu_df']}
active = {k:v for k,v in named.items() if v is not None}

if not active:
    print('No datasets loaded. Run Cell 3 first.')
else:
    for dname, df in active.items():
        hdr(f'{dname}  —  {df.shape[0]:,} rows × {df.shape[1]} cols')
        display(df.head(5))
        for tc in ['Fault_Type','fault_type','Fault_Type_Num']:
            if tc in df.columns:
                vc = df[tc].value_counts().sort_index()
                fig, ax = plt.subplots(figsize=(8,3))
                ax.bar(vc.index.astype(str), vc.values,
                       color=PALETTE[:len(vc)], width=0.55)
                ax.set_title(f'{dname} — Class Distribution ({tc})')
                ax.set_xlabel('Class'); ax.set_ylabel('Count'); ax.grid(axis='y',alpha=0.4)
                plt.tight_layout(); plt.show(); plt.close()
                break
        nulls = df.isnull().sum(); nulls = nulls[nulls>0]
        if nulls.empty: print(f'  No null values ✓\n')
        else: print('  Nulls:'); display(nulls)
print('✅ Preview complete')


## Cell 5 — Feature configuration (interactive widgets)

In [ ]:
ALL_ELEC = ['Ia','Ib','Ic','Va','Vb','Vc']
src = STATE['train_df'] if STATE['train_df'] is not None else STATE['pmu_df']
if src is None:
    raise RuntimeError('Upload a training dataset first (Cell 3).')

avail_elec = [f for f in ALL_ELEC if f in src.columns]
target_col = None
for tc in ['Fault_Type','fault_type','Fault_Type_Num']:
    if tc in src.columns: target_col = tc; break
if target_col is None:
    raise RuntimeError('No target column found (expected Fault_Type / fault_type).')

STATE['target_col'] = target_col
print(f'Target  : {target_col}')
print(f'Features: {avail_elec}')

hdr('Select Features & Hyperparameters')
feat_select = widgets.SelectMultiple(
    options=avail_elec, value=avail_elec[:4] if len(avail_elec)>=4 else avail_elec,
    description='Features:', rows=len(avail_elec),
    style={'description_width':'80px'}, layout=widgets.Layout(width='230px'))
n_est_w = widgets.IntSlider(value=200, min=50, max=400, step=50,
    description='n_estimators:',
    style={'description_width':'110px'}, layout=widgets.Layout(width='380px'))
depth_w = widgets.Dropdown(options=[5,10,15,20,30,None], value=20,
    description='max_depth:', style={'description_width':'90px'},
    layout=widgets.Layout(width='220px'))
test_sz = widgets.FloatSlider(value=0.2, min=0.1, max=0.4, step=0.05,
    description='Val split:', readout_format='.0%',
    style={'description_width':'80px'}, layout=widgets.Layout(width='360px'))

display(widgets.HBox([feat_select,
    widgets.VBox([n_est_w, depth_w, test_sz],
                 layout=widgets.Layout(margin='0 0 0 20px'))]))
print('\n(Ctrl/Cmd-click for multi-select). Then run Cell 6.')


## Cell 6 — Feature analysis: Correlation + VIF + RFE

In [ ]:
feats = list(feat_select.value)
if not feats:
    raise RuntimeError('Select at least one feature in Cell 5.')
STATE['features'] = feats
print(f'Features: {feats}')

df_src = STATE['train_df'] if STATE['train_df'] is not None else STATE['pmu_df']
X_all  = df_src[feats].dropna()

# ── Correlation + VIF ─────────────────────────────────────────────
if len(feats) >= 2:
    hdr('Correlation Heatmap with VIF on Diagonal')
    corr = X_all.corr().round(4)
    try:
        X_c  = add_constant(X_all.astype(float))
        vifs = [round(variance_inflation_factor(X_c.values, i+1), 2)
                for i in range(len(feats))]
    except Exception:
        vifs = ['N/A'] * len(feats)
    fig, ax = plt.subplots(figsize=(max(7, len(feats)*1.6), max(5, len(feats)*1.4)))
    sns.heatmap(corr, annot=True, fmt='.4f', cmap='RdBu_r',
                vmin=-1, vmax=1, linewidths=0.5, linecolor='#1a1a2e',
                ax=ax, cbar_kws={'shrink':0.8})
    for i, feat in enumerate(feats):
        ax.text(i+0.5, i+0.5, f'VIF\n{vifs[i]}',
                ha='center', va='center', fontsize=9, fontweight='bold',
                bbox=dict(boxstyle='square,pad=0.35', facecolor='#0f0f2d',
                          edgecolor='#00d4ff', alpha=0.9), color='#00d4ff')
    ax.set_title('Correlation Matrix with VIF (diagonal)')
    ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha='right')
    plt.tight_layout(); plt.show(); plt.close()

# ── Quick RF importance ───────────────────────────────────────────
hdr('Feature Importance — Random Forest (quick fit)')
tc  = STATE['target_col']
y_q = df_src[tc]
if y_q.dtype == object:
    y_q = pd.Series(LabelEncoder().fit_transform(y_q), index=y_q.index)
rf_q = RandomForestClassifier(n_estimators=100, max_depth=15, random_state=42, n_jobs=-1)
rf_q.fit(X_all.values, y_q.loc[X_all.index].values)
imp_df = pd.DataFrame({'Feature':feats,'Importance':rf_q.feature_importances_*100})\
           .sort_values('Importance', ascending=True)
fig, ax = plt.subplots(figsize=(9, max(3, len(feats)*0.7)))
bars = ax.barh(imp_df['Feature'], imp_df['Importance'],
               color=[PALETTE[i%len(PALETTE)] for i in range(len(imp_df))], height=0.55)
for bar, val in zip(bars, imp_df['Importance']):
    ax.text(bar.get_width()+0.2, bar.get_y()+bar.get_height()/2,
            f'{val:.1f}%', va='center', fontsize=10)
ax.set_xlabel('Importance (%)'); ax.set_title('Feature Importances')
ax.grid(axis='x', alpha=0.4)
plt.tight_layout(); plt.show(); plt.close()

# ── RFE on synthetic data ─────────────────────────────────────────
if STATE['synthetic_df'] is not None:
    hdr('RFE Feature Ranking — Historical Data')
    sdf = STATE['synthetic_df'].copy()
    fault_cols = [c for c in ['L_G_Fault','L_L_Fault','L_L_G_Fault',
                               'LLL_LLLG_Fault','Open_Circuits','Insulation_Failure']
                  if c in sdf.columns]
    if fault_cols:
        sdf['Dom']  = sdf[fault_cols].idxmax(axis=1)
        sdf['DomN'] = sdf[fault_cols].max(axis=1)
        fm2 = {'L_G_Fault':2,'L_L_Fault':3,'L_L_G_Fault':4,
               'LLL_LLLG_Fault':5,'Open_Circuits':5,'Insulation_Failure':5}
        sdf['FT'] = sdf['Dom'].map(fm2)
        sdf.loc[sdf['DomN']==0,'FT'] = 0
        for ec in ['Month','Feeder']:
            if ec in sdf.columns:
                sdf[ec+'_Enc'] = LabelEncoder().fit_transform(sdf[ec].astype(str))
        rfe_feats = [c for c in ['Year','Month_Enc','Feeder_Enc','Weather_Factor',
                                  'Fault_Location_km','Fault_Duration_s'] if c in sdf.columns]
        if len(rfe_feats) >= 2:
            n_sel = min(4, len(rfe_feats))
            rfe   = RFE(RandomForestClassifier(n_estimators=80, random_state=42),
                        n_features_to_select=n_sel)
            rfe.fit(sdf[rfe_feats].fillna(0), sdf['FT'])
            rdf = pd.DataFrame({'Feature':rfe_feats,'Ranking':rfe.ranking_,
                                 'Selected':rfe.support_}).sort_values('Ranking')
            fig, ax = plt.subplots(figsize=(9, max(3, len(rfe_feats)*0.7)))
            colors_r = ['#00d4ff' if s else '#444466' for s in rdf['Selected']]
            bars = ax.barh(rdf['Feature'], rdf['Ranking'], color=colors_r, height=0.55)
            for i, bar in enumerate(bars):
                ax.text(bar.get_width()+0.04, bar.get_y()+bar.get_height()/2,
                        f"Rank {rdf['Ranking'].iloc[i]}", va='center', fontsize=10)
            ax.set_xlabel('RFE Ranking (1 = most important)')
            ax.set_title(f'RFE Feature Selection (top {n_sel} in blue)')
            ax.grid(axis='x', alpha=0.4)
            plt.tight_layout(); plt.show(); plt.close()
            print(rdf.to_string(index=False))

print('\n✅ Feature analysis complete')


## Cell 7 — Train Random Forest + evaluate

In [ ]:
feats    = STATE['features']
tc       = STATE['target_col']
df_train = (STATE['train_df'] if STATE['train_df'] is not None else STATE['pmu_df']).copy()

X = df_train[feats].fillna(0)
y = df_train[tc]
if y.dtype == object:
    le = LabelEncoder(); y = pd.Series(le.fit_transform(y), index=y.index)
    STATE['label_enc'] = le; print(f'Classes: {dict(enumerate(le.classes_))}')
else:
    STATE['label_enc'] = None

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=test_sz.value, random_state=42, stratify=y)

scaler = MinMaxScaler()
X_tr_s = scaler.fit_transform(X_train)
X_va_s = scaler.transform(X_val)
STATE['scaler'] = scaler

hdr('Training Random Forest...')
rf = RandomForestClassifier(n_estimators=n_est_w.value, max_depth=depth_w.value,
                             random_state=42, n_jobs=-1)
rf.fit(X_tr_s, y_train)
STATE['model'] = rf; STATE['X_val'] = X_va_s
STATE['y_val'] = y_val

y_pred = rf.predict(X_va_s); STATE['y_pred'] = y_pred
acc    = accuracy_score(y_val, y_pred)
rpt    = classification_report(y_val, y_pred, output_dict=True)
kpi_row({'Accuracy': f'{acc*100:.2f}%',
         'Macro F1': f"{rpt['macro avg']['f1-score']:.4f}",
         'Precision': f"{rpt['weighted avg']['precision']:.4f}",
         'Recall':    f"{rpt['weighted avg']['recall']:.4f}"})

print('\nClassification Report:')
print(classification_report(y_val, y_pred, digits=4))

hdr('Confusion Matrix')
classes = sorted(y_val.unique())
cm = confusion_matrix(y_val, y_pred, labels=classes)
fig, ax = plt.subplots(figsize=(8,6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=classes, yticklabels=classes,
            ax=ax, linewidths=0.4, linecolor='#1a1a2e')
ax.set_xlabel('Predicted'); ax.set_ylabel('Actual')
ax.set_title('Confusion Matrix — Validation Set')
plt.tight_layout(); plt.show(); plt.close()
print('✅ Model trained and evaluated')


## Cell 8 — ROC Curves + Learning Curve

In [ ]:
rf = STATE['model']; X_va_s = STATE['X_val']; y_val = STATE['y_val']
if rf is None: raise RuntimeError('Train the model first (Cell 7).')

hdr('ROC Curves — One-vs-Rest')
classes_list = sorted(y_val.unique())
y_bin  = label_binarize(y_val, classes=classes_list)
y_prob = rf.predict_proba(X_va_s)

fig, ax = plt.subplots(figsize=(10,7))
for i, (cls, col) in enumerate(zip(classes_list, cycle(PALETTE))):
    if i < y_bin.shape[1] and i < y_prob.shape[1]:
        fpr, tpr, _ = roc_curve(y_bin[:,i], y_prob[:,i])
        ax.plot(fpr, tpr, color=col, lw=2,
                label=f"{LABEL_MAP.get(cls,f'Class {cls}')}  (AUC={auc(fpr,tpr):.3f})")
ax.plot([0,1],[0,1],'w--',lw=1.2,alpha=0.5)
ax.set_xlabel('False Positive Rate'); ax.set_ylabel('True Positive Rate')
ax.set_title('ROC Curves'); ax.legend(fontsize=9); ax.grid(alpha=0.35)
plt.tight_layout(); plt.show(); plt.close()

hdr('Learning Curve Analysis')
print('Computing (CV=3, may take ~30 s)...')
feats = STATE['features']; tc = STATE['target_col']
df_tr = (STATE['train_df'] if STATE['train_df'] is not None else STATE['pmu_df']).copy()
X_lc  = STATE['scaler'].transform(df_tr[feats].fillna(0))
y_lc  = df_tr[tc]
if y_lc.dtype == object and STATE['label_enc'] is not None:
    y_lc = STATE['label_enc'].transform(y_lc)

lc_rf = RandomForestClassifier(n_estimators=n_est_w.value, max_depth=depth_w.value,
                                random_state=42, n_jobs=-1)
sizes, tr_sc, va_sc = learning_curve(lc_rf, X_lc, y_lc,
    train_sizes=np.linspace(0.1,1.0,6), cv=3, scoring='accuracy', n_jobs=-1)
tm,ts_ = tr_sc.mean(1),tr_sc.std(1); vm,vs_ = va_sc.mean(1),va_sc.std(1)
fig, ax = plt.subplots(figsize=(10,5))
ax.plot(sizes,tm,'o-',color='#f72585',lw=2,label='Training score')
ax.fill_between(sizes,tm-ts_,tm+ts_,alpha=0.15,color='#f72585')
ax.plot(sizes,vm,'o-',color='#4cc9f0',lw=2,label='CV score')
ax.fill_between(sizes,vm-vs_,vm+vs_,alpha=0.15,color='#4cc9f0')
ax.set_xlabel('Training Set Size'); ax.set_ylabel('Accuracy')
ax.set_title('Learning Curve — Random Forest')
ax.legend(); ax.grid(alpha=0.35)
plt.tight_layout(); plt.show(); plt.close()
print('✅ ROC + Learning Curve complete')


## Cell 9 — Select prediction source

In [ ]:
if STATE['model'] is None:
    raise RuntimeError('Train the model first (Cell 7).')

pred_options = {}
if STATE['test_df']  is not None: pred_options['Test Dataset']            = STATE['test_df']
if STATE['pmu_df']   is not None: pred_options['PMU / Raw Dataset']       = STATE['pmu_df']
if STATE['train_df'] is not None: pred_options['Training Data (self-check)'] = STATE['train_df']

if not pred_options:
    raise RuntimeError('No dataset available. Upload one in Cell 3.')

hdr('Choose Dataset to Predict On')
src_selector = widgets.ToggleButtons(
    options=list(pred_options.keys()),
    description='Predict on:',
    style={'description_width':'90px','button_width':'220px'})
display(src_selector)
print('Select above, then run Cell 10.')


## Cell 10 — Execute predictions & summary plots

In [ ]:
feats  = STATE['features']
scaler = STATE['scaler']
rf     = STATE['model']
src_df = pred_options[src_selector.value].copy()
avail  = [f for f in feats if f in src_df.columns]
if not avail:
    raise RuntimeError(f'None of {feats} found in selected dataset.')

X_pred = scaler.transform(src_df[avail].fillna(0))
preds  = rf.predict(X_pred)
probas = rf.predict_proba(X_pred)

src_df['Predicted_Fault_Type'] = preds
src_df['Confidence_%'] = (probas.max(axis=1)*100).round(1)
if STATE['label_enc'] is not None:
    src_df['Fault_Label'] = STATE['label_enc'].inverse_transform(preds)
else:
    src_df['Fault_Label'] = [LABEL_MAP.get(int(p), str(p)) for p in preds]

if 'Timestamp' not in src_df.columns:
    t0 = datetime(2025,4,1)
    src_df['Timestamp'] = [t0+timedelta(minutes=15*i) for i in range(len(src_df))]

STATE['predictions'] = src_df

total = len(src_df)
f_cnt = (src_df['Fault_Label'].str.contains('Fault',case=False)).sum()
n_cnt = total - f_cnt
f_rt  = 100*f_cnt/total if total>0 else 0
avg_c = src_df['Confidence_%'].mean()
kpi_row({'Total Records':f'{total:,}','Fault Events':f'{f_cnt:,}',
         'Normal':f'{n_cnt:,}','Fault Rate':f'{f_rt:.1f}%','Avg Conf.':f'{avg_c:.1f}%'})

hdr('Predicted Fault Distribution')
vc = src_df['Fault_Label'].value_counts()
fig, ax = plt.subplots(figsize=(10,max(3,len(vc)*0.6)))
clrs = ['#f72585' if 'Fault' in str(v) else '#00d4ff' for v in vc.index]
bars = ax.barh(vc.index, vc.values, color=clrs, height=0.55)
for bar,val in zip(bars,vc.values):
    ax.text(bar.get_width()+max(vc.values)*0.01,
            bar.get_y()+bar.get_height()/2, f'{val:,}', va='center', fontsize=10)
ax.set_xlabel('Count'); ax.set_title('Predicted Fault Type Breakdown')
ax.grid(axis='x',alpha=0.4); plt.tight_layout(); plt.show(); plt.close()

sig_cols = [f for f in ['Ia','Ib','Ic','Va','Vb','Vc'] if f in src_df.columns]
if sig_cols:
    hdr('Electrical Signal Overview (first 500 samples)')
    nc = min(len(sig_cols),4)
    fig,axes = plt.subplots(1,nc,figsize=(14,3.5))
    if nc==1: axes=[axes]
    for ax,col in zip(axes,sig_cols[:nc]):
        ax.plot(src_df[col].values[:500],lw=0.8,color='#00d4ff',alpha=0.85)
        ax.set_title(col); ax.grid(alpha=0.3); ax.tick_params(axis='x',labelsize=7)
    fig.suptitle('Electrical Signals',y=1.02); plt.tight_layout(); plt.show(); plt.close()

hdr('Fault Frequency Over Time')
src_df['Date'] = pd.to_datetime(src_df['Timestamp']).dt.date
trend = src_df.groupby(['Date','Fault_Label']).size().unstack(fill_value=0)
fig, ax = plt.subplots(figsize=(14,5))
for col_t in trend.columns:
    c = '#f72585' if 'Fault' in str(col_t) else '#4cc9f0'
    ax.plot(trend.index.astype(str), trend[col_t].values, label=str(col_t), lw=1.6, color=c)
step = max(1,len(trend)//10)
ax.set_xticks(range(0,len(trend),step))
ax.set_xticklabels([str(trend.index[i]) for i in range(0,len(trend),step)],
                   rotation=35,ha='right',fontsize=8)
ax.set_title('Fault Frequency Over Time'); ax.legend(fontsize=9); ax.grid(alpha=0.35)
plt.tight_layout(); plt.show(); plt.close()

show_cols = (['Timestamp']+sig_cols[:4]+['Predicted_Fault_Type','Fault_Label','Confidence_%'])
show_cols = [c for c in show_cols if c in src_df.columns]
hdr('Predictions Table — first 20 rows'); display(src_df[show_cols].head(20))
print(f'\n✅ {total:,} predictions generated from {src_selector.value}')


## Cell 11 — SHAP Explainability

In [ ]:
rf = STATE['model']; X_va_s = STATE['X_val']; feats = STATE['features']
if rf is None: raise RuntimeError('Train the model first (Cell 7).')

hdr('SHAP Feature Explanation — Random Sample')
print('Computing SHAP values (this may take ~20s)...')

bg_size   = min(100, len(X_va_s))
bg        = shap.sample(X_va_s, bg_size, random_state=42)
explainer = shap.TreeExplainer(rf, data=bg)

idx         = np.random.randint(0, len(X_va_s))
sample      = X_va_s[idx:idx+1]
pred_class  = rf.predict(sample)[0]
pred_label  = LABEL_MAP.get(int(pred_class), str(pred_class))
sv          = explainer.shap_values(sample)

if isinstance(sv, list):
    vals = sv[int(pred_class)%len(sv)].flatten()
elif len(np.array(sv).shape) == 3:
    vals = np.array(sv)[0,:,int(pred_class)%np.array(sv).shape[2]]
else:
    vals = np.array(sv).flatten()
vals = vals[:len(feats)]

fig, (ax1,ax2) = plt.subplots(1,2,figsize=(14,5))
ax1.plot(feats, sample.flatten(), 'o-', color='#00d4ff', lw=2, ms=8)
ax1.set_title(f'Input Waveform — Predicted: {pred_label}')
ax1.set_xlabel('Feature'); ax1.set_ylabel('Scaled Value'); ax1.grid(alpha=0.4)
for xi,yi in enumerate(sample.flatten()):
    ax1.annotate(f'{yi:.3f}',(xi,yi),textcoords='offset points',
                 xytext=(0,10),ha='center',fontsize=8,color='#c8c8e8')

clrs_s = ['#f72585' if v>0 else '#4cc9f0' for v in vals]
bars   = ax2.barh(feats, vals, color=clrs_s, height=0.55)
ax2.axvline(0,color='white',lw=0.8,alpha=0.5)
ax2.set_xlabel('SHAP value')
ax2.set_title(f'SHAP Explanation — Class: {pred_label}'); ax2.grid(axis='x',alpha=0.4)
for bar,val in zip(bars,vals):
    offset = max(abs(vals))*0.03
    ax2.text(val+(offset if val>=0 else -offset),
             bar.get_y()+bar.get_height()/2, f'{val:.4f}',
             va='center', ha='left' if val>=0 else 'right', fontsize=9)

action = 'TRIP ⚠️' if pred_class in FAULT_CLASSES else 'BLOCK ✅'
fig.suptitle(f'Relay Action: {action}', fontsize=14,
             color='#f72585' if 'TRIP' in action else '#4cc9f0', y=1.02)
plt.tight_layout(); plt.show(); plt.close()
print(f'  Predicted class : {pred_class} ({pred_label})\n  Action : {action}')


## Cell 12 — Interactive SCADA dashboard

In [ ]:
pred_df = STATE['predictions']
if pred_df is None:
    raise RuntimeError('Run predictions first (Cells 9–10).')

pred_df['Timestamp'] = pd.to_datetime(pred_df['Timestamp'])
all_labels = sorted(pred_df['Fault_Label'].unique())
sig_cols   = [f for f in ['Ia','Ib','Ic','Va','Vb','Vc'] if f in pred_df.columns]

date_opts = pd.date_range(pred_df['Timestamp'].min().date(),
                           pred_df['Timestamp'].max().date(), freq='D').date.tolist()
date_range = widgets.SelectionRangeSlider(
    options=date_opts, index=(0,len(date_opts)-1),
    description='Date range:',
    style={'description_width':'90px'}, layout=widgets.Layout(width='600px'))
label_filter = widgets.SelectMultiple(
    options=all_labels, value=all_labels, description='Fault types:',
    rows=min(6,len(all_labels)),
    style={'description_width':'90px'}, layout=widgets.Layout(width='280px'))
plot_type = widgets.ToggleButtons(
    options=['Distribution','Trend','Signals','All'], value='All',
    description='Show:',
    style={'description_width':'50px','button_width':'110px'})
refresh_btn  = widgets.Button(description='🔄 Refresh',  button_style='info',
                               layout=widgets.Layout(width='150px',height='36px'))
download_btn = widgets.Button(description='⬇ Download CSV', button_style='success',
                               layout=widgets.Layout(width='160px',height='36px'))
out = widgets.Output()

def render(_=None):
    with out:
        clear_output(wait=True)
        sd,ed = date_range.value
        sel   = list(label_filter.value)
        mask  = ((pred_df['Timestamp'].dt.date>=sd) &
                 (pred_df['Timestamp'].dt.date<=ed) &
                 (pred_df['Fault_Label'].isin(sel)))
        fdf   = pred_df[mask].copy()
        if fdf.empty: print('No data for current filters.'); return

        total = len(fdf)
        f_cnt = (fdf['Fault_Label'].str.contains('Fault',case=False)).sum()
        f_rt  = 100*f_cnt/total
        avg_c = fdf['Confidence_%'].mean() if 'Confidence_%' in fdf.columns else 0
        kpi_row({'Records':f'{total:,}','Faults':f'{f_cnt:,}',
                 'Normal':f'{total-f_cnt:,}','Fault Rate':f'{f_rt:.1f}%',
                 'Avg Conf.':f'{avg_c:.1f}%'})

        pt = plot_type.value
        if pt in ('Distribution','All'):
            vc = fdf['Fault_Label'].value_counts()
            fig,ax = plt.subplots(figsize=(10,max(3,len(vc)*0.6)))
            clrs = ['#f72585' if 'Fault' in str(v) else '#00d4ff' for v in vc.index]
            bars = ax.barh(vc.index,vc.values,color=clrs,height=0.5)
            for bar,val in zip(bars,vc.values):
                ax.text(bar.get_width()+max(vc.values)*0.01,
                        bar.get_y()+bar.get_height()/2,f'{val:,}',va='center',fontsize=10)
            ax.set_title('Fault Type Distribution'); ax.set_xlabel('Count')
            ax.grid(axis='x',alpha=0.4); plt.tight_layout(); plt.show(); plt.close()

        if pt in ('Trend','All'):
            fdf['Date'] = fdf['Timestamp'].dt.date
            trend = fdf.groupby(['Date','Fault_Label']).size().unstack(fill_value=0)
            fig,ax = plt.subplots(figsize=(14,5))
            for col_t in trend.columns:
                c = '#f72585' if 'Fault' in str(col_t) else '#4cc9f0'
                ax.plot(trend.index.astype(str),trend[col_t].values,label=str(col_t),lw=1.5,color=c)
            step = max(1,len(trend)//8)
            ax.set_xticks(range(0,len(trend),step))
            ax.set_xticklabels([str(trend.index[i]) for i in range(0,len(trend),step)],
                               rotation=35,ha='right',fontsize=8)
            ax.set_title('Fault Frequency Over Time')
            ax.legend(fontsize=9); ax.grid(alpha=0.35)
            plt.tight_layout(); plt.show(); plt.close()

        if pt in ('Signals','All') and sig_cols:
            nc = min(len(sig_cols),4)
            fig,axes = plt.subplots(1,nc,figsize=(14,3.5))
            if nc==1: axes=[axes]
            for ax,col in zip(axes,sig_cols[:nc]):
                ax.plot(fdf[col].values[:500],lw=0.8,color='#00d4ff',alpha=0.85)
                ax.set_title(col); ax.grid(alpha=0.3); ax.tick_params(axis='x',labelsize=7)
            fig.suptitle('Electrical Signals (first 500 samples)',y=1.02)
            plt.tight_layout(); plt.show(); plt.close()

        bd = (fdf['Fault_Label'].value_counts().rename_axis('Fault Type').reset_index(name='Count'))
        bd['%'] = (100*bd['Count']/total).round(2)
        display(HTML('<b>Fault Breakdown Table</b>')); display(bd)

def download_csv(_):
    sd,ed = date_range.value; sel = list(label_filter.value)
    mask  = ((pred_df['Timestamp'].dt.date>=sd) &
             (pred_df['Timestamp'].dt.date<=ed) &
             (pred_df['Fault_Label'].isin(sel)))
    pred_df[mask].to_csv('filtered_predictions.csv',index=False)
    files.download('filtered_predictions.csv')

refresh_btn.on_click(render); download_btn.on_click(download_csv)

display(HTML(
    "<div style='background:#0f0f2d;border:1px solid #00d4ff33;border-radius:8px;"
    "padding:12px 18px;font-family:monospace;color:#00d4ff;font-size:14px;'>"
    "⚡ SCADA Dashboard — Fault Monitoring</div>"))
display(widgets.VBox([
    widgets.HBox([date_range]),
    widgets.HBox([label_filter,
                  widgets.VBox([plot_type, widgets.HBox([refresh_btn,download_btn])],
                               layout=widgets.Layout(margin='0 0 0 20px'))]),
    out]))
render()


## Cell 13 — Export results

In [ ]:
hdr('Export Results')
if STATE['predictions'] is not None:
    STATE['predictions'].to_csv('all_predictions.csv', index=False)
    files.download('all_predictions.csv')
    print('✅ all_predictions.csv downloaded')

if STATE['y_val'] is not None and STATE['y_pred'] is not None:
    rpt = classification_report(STATE['y_val'], STATE['y_pred'], output_dict=True)
    pd.DataFrame(rpt).T.round(4).to_csv('classification_report.csv')
    files.download('classification_report.csv')
    print('✅ classification_report.csv downloaded')

print('Done.')
